# 🔥 Predicting Time-to-Threat for Wildfire Evacuation Zones
**WiDS Global Datathon 2026 · Survival Analysis**

Predicts, from only the first **5 hours** of a wildfire, the probability it threatens an evacuation zone
within **12 / 24 / 48 / 72 hours**. Pipeline: engineered features → survival ensemble
(XGBoost Cox + Random Survival Forest + Gradient Boosted Survival Trees) → **per-horizon calibration** →
physical guardrails.

> **Self-contained Kaggle version.** This notebook is auto-generated from the modular, unit-tested source
> in our GitHub repo (`src/`) by `scripts/build_kaggle_notebook.py` — that repo is the canonical source.
> Baseline **0.87397 → 0.96366** after these enhancements.

### Install dependencies (Kaggle kernels don't ship scikit-survival)

In [ ]:
!pip install -q scikit-survival optuna xgboost

### Imports

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from itertools import product
from sklearn.model_selection import KFold
from sksurv.metrics import concordance_index_censored
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split
from pathlib import Path

### Configuration
Set `DATA_DIR` to the competition's input folder, and confirm the column names match the real dataset
headers (`ID_COL`, `EVENT_COL`, `TIME_COL`, `RAW_FEATURES`).

In [ ]:
from types import SimpleNamespace
from pathlib import Path

config = SimpleNamespace(
    # ---- paths (Kaggle) ----
    DATA_DIR=Path("/kaggle/input/WiDSWorldWide_GlobalDathon26"),   # <-- confirm slug
    SUBMISSIONS_DIR=Path("/kaggle/working"),
    TRAIN_FILE="train.csv",
    TEST_FILE="test.csv",
    # ---- problem definition ----
    HORIZONS=(12, 24, 48, 72),
    HIT_RADIUS_KM=5.0,
    FEATURE_WINDOW_HOURS=5,
    # ---- columns (ALIGN WITH REAL DATA) ----
    ID_COL="event_id",
    EVENT_COL="event",
    TIME_COL="time_to_hit_hours",
    RAW_FEATURES=[
        "distance_to_zone_km", "spread_rate_kmh", "perimeter_growth_km2",
        "bearing_to_zone_deg", "wind_speed_kmh", "wind_alignment",
        "fuel_dryness_index", "terrain_slope",
    ],
    RANDOM_STATE=42,
    N_SPLITS=5,
)

### Feature engineering (+20 features)
Distance-decay, distance×speed interactions, directional, growth-dynamics, composite risk scores.

In [ ]:
EPS = 1e-6


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of ``df`` with the full engineered feature set appended.

    Grouped by intent so the reasoning is legible. Guards with ``.get`` keep the
    function robust if a raw column is renamed or missing in your copy of the
    data — engineer only what the inputs support.
    """
    df = df.copy()
    df = _add_distance_decay_features(df)
    df = _add_distance_speed_interactions(df)
    df = _add_directional_features(df)
    df = _add_growth_dynamics_features(df)
    df = _add_composite_risk_scores(df)
    return df


# --------------------------------------------------------------------------- #
# 1. Distance-decay — the direct fix for distant-fire overconfidence
# --------------------------------------------------------------------------- #
def _add_distance_decay_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.get("distance_to_zone_km")
    if d is None:
        return df
    # Sharp, monotone decay so the model can express "risk ~ 0 beyond a few km".
    df["dist_inverse"] = 1.0 / (d + 1.0)
    df["dist_exp_decay_2km"] = np.exp(-d / 2.0)
    df["dist_exp_decay_5km"] = np.exp(-d / config.HIT_RADIUS_KM)
    df["dist_log1p"] = np.log1p(d)
    df["dist_sq"] = d ** 2
    # Hard indicators around the 5 km hit radius — a strong, interpretable prior.
    df["within_hit_radius"] = (d <= config.HIT_RADIUS_KM).astype(int)
    df["within_2x_hit_radius"] = (d <= 2 * config.HIT_RADIUS_KM).astype(int)
    df["is_distant_fire"] = (d > 4 * config.HIT_RADIUS_KM).astype(int)
    return df


# --------------------------------------------------------------------------- #
# 2. Distance x speed interactions — "how fast is it closing the gap?"
# --------------------------------------------------------------------------- #
def _add_distance_speed_interactions(df: pd.DataFrame) -> pd.DataFrame:
    d = df.get("distance_to_zone_km")
    v = df.get("spread_rate_kmh")
    if d is None or v is None:
        return df
    # Naive time-to-reach if the fire drove straight at the zone at current speed.
    df["hours_to_zone_naive"] = d / (v + EPS)
    df["speed_per_distance"] = v / (d + 1.0)          # closing "pressure"
    df["speed_x_proximity"] = v * np.exp(-d / config.HIT_RADIUS_KM)
    # Can the fire physically cover the distance within each horizon?
    for h in config.HORIZONS:
        df[f"reachable_by_{h}h"] = (v * h >= d).astype(int)
    return df


# --------------------------------------------------------------------------- #
# 3. Directional features — a fast fire pointed *away* is not a threat
# --------------------------------------------------------------------------- #
def _add_directional_features(df: pd.DataFrame) -> pd.DataFrame:
    align = df.get("wind_alignment")           # cos(angle) toward the zone, [-1, 1]
    v = df.get("spread_rate_kmh")
    if align is not None and v is not None:
        # Effective speed *component* aimed at the zone.
        df["directed_speed"] = v * align.clip(lower=0)
        df["heading_toward_zone"] = (align > 0).astype(int)
    return df


# --------------------------------------------------------------------------- #
# 4. Growth dynamics — acceleration/energy signals from the 5h window
# --------------------------------------------------------------------------- #
def _add_growth_dynamics_features(df: pd.DataFrame) -> pd.DataFrame:
    area = df.get("perimeter_growth_km2")
    if area is not None:
        df["growth_rate_per_hour"] = area / config.FEATURE_WINDOW_HOURS
        df["log_growth"] = np.log1p(area)
    dry = df.get("fuel_dryness_index")
    wind = df.get("wind_speed_kmh")
    if dry is not None and wind is not None:
        # A simple fire-weather energy proxy: dry fuel + wind => faster spread.
        df["fire_weather_energy"] = dry * wind
    return df


# --------------------------------------------------------------------------- #
# 5. Composite risk scores — bundle the signals into interpretable indices
# --------------------------------------------------------------------------- #
def _add_composite_risk_scores(df: pd.DataFrame) -> pd.DataFrame:
    d = df.get("distance_to_zone_km")
    v = df.get("spread_rate_kmh")
    if d is None or v is None:
        return df
    proximity = np.exp(-d / config.HIT_RADIUS_KM)                 # 1 when on top of zone -> 0 far away
    speed_norm = (v - v.min()) / (v.max() - v.min() + EPS)        # 0..1
    directed = df.get("directed_speed", v)
    directed_norm = (directed - directed.min()) / (directed.max() - directed.min() + EPS)

    # Overall "threat" index and a distance-gated version (0 for distant fires).
    df["risk_score"] = proximity * (0.5 + 0.5 * speed_norm)
    df["risk_score_directed"] = proximity * (0.5 + 0.5 * directed_norm)
    df["gated_risk"] = df["risk_score"] * df.get("within_2x_hit_radius", 1)
    return df


def engineered_feature_columns(df: pd.DataFrame) -> list[str]:
    """All model-input columns: engineered features plus any raw features present.

    Excludes the id and survival-label columns.
    """
    exclude = {config.ID_COL, config.EVENT_COL, config.TIME_COL}
    return [c for c in df.columns if c not in exclude]

### Survival models
XGBoost Cox (with a Breslow baseline), Random Survival Forest, Gradient Boosted Survival Trees — behind one `predict_hit_prob` interface. Includes optional Optuna tuning.

In [ ]:
# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #
def to_structured_y(event: np.ndarray, time: np.ndarray) -> np.ndarray:
    """Build the (bool event, float time) structured array scikit-survival wants."""
    y = np.empty(len(event), dtype=[("event", bool), ("time", float)])
    y["event"] = event.astype(bool)
    y["time"] = time.astype(float)
    return y


def _survfuncs_to_hit_prob(surv_funcs, horizons) -> np.ndarray:
    """Evaluate a list of sksurv step-function survival curves at each horizon.

    Returns 1 - S(h) = P(hit by h). Values are clipped to [0, 1].
    """
    out = np.empty((len(surv_funcs), len(horizons)))
    for i, fn in enumerate(surv_funcs):
        for j, h in enumerate(horizons):
            out[i, j] = 1.0 - float(fn(h))
    return np.clip(out, 0.0, 1.0)


# --------------------------------------------------------------------------- #
# 1. XGBoost Cox Proportional Hazards
# --------------------------------------------------------------------------- #
@dataclass
class XGBoostCoxModel:
    """XGBoost with the Cox partial-likelihood objective.

    XGBoost's `survival:cox` outputs a *relative risk* (exp of the log-hazard),
    not an absolute survival probability. To turn risk into P(hit by h) we fit a
    non-parametric **Breslow** baseline cumulative hazard H0(t) on the training
    data, then use the proportional-hazards form:
        S(t | x) = exp( -H0(t) * risk(x) )
        P(hit by t | x) = 1 - S(t | x)
    """
    params: dict = field(default_factory=lambda: {
        "objective": "survival:cox",
        "eval_metric": "cox-nloglik",
        "learning_rate": 0.03,
        "max_depth": 3,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 2.0,
        "reg_alpha": 0.5,
        "tree_method": "hist",
        "seed": config.RANDOM_STATE,
    })
    n_estimators: int = 400
    _booster: object = field(default=None, repr=False)
    _base_times: np.ndarray = field(default=None, repr=False)
    _base_cumhaz: np.ndarray = field(default=None, repr=False)

    def fit(self, X, event, time):
        import xgboost as xgb

        # XGBoost encodes censoring via the sign of the label: negative time =
        # censored, positive time = observed event.
        signed_time = np.where(event.astype(bool), time, -time)
        dtrain = xgb.DMatrix(np.asarray(X, dtype=float), label=signed_time)
        self._booster = xgb.train(self.params, dtrain, num_boost_round=self.n_estimators)

        # Breslow baseline cumulative hazard from training risk scores.
        risk = np.exp(self._booster.predict(dtrain))
        self._fit_breslow_baseline(time, event, risk)
        return self

    def _fit_breslow_baseline(self, time, event, risk):
        order = np.argsort(time)
        t_sorted, e_sorted, r_sorted = time[order], event[order], risk[order]
        uniq_event_times = np.unique(t_sorted[e_sorted.astype(bool)])
        cumhaz, running = [], 0.0
        for t in uniq_event_times:
            d = np.sum((t_sorted == t) & e_sorted.astype(bool))   # ties
            at_risk = np.sum(t_sorted >= t)
            denom = r_sorted[t_sorted >= t].sum()
            running += d / (denom + 1e-12)
            cumhaz.append(running)
        self._base_times = uniq_event_times
        self._base_cumhaz = np.asarray(cumhaz)

    def _baseline_cumhaz_at(self, h: float) -> float:
        if self._base_times is None or len(self._base_times) == 0:
            return 0.0
        idx = np.searchsorted(self._base_times, h, side="right") - 1
        return 0.0 if idx < 0 else float(self._base_cumhaz[idx])

    def predict_hit_prob(self, X, horizons=config.HORIZONS) -> np.ndarray:
        import xgboost as xgb

        risk = np.exp(self._booster.predict(xgb.DMatrix(np.asarray(X, dtype=float))))
        out = np.empty((len(risk), len(horizons)))
        for j, h in enumerate(horizons):
            H0 = self._baseline_cumhaz_at(h)
            out[:, j] = 1.0 - np.exp(-H0 * risk)
        return np.clip(out, 0.0, 1.0)


# --------------------------------------------------------------------------- #
# 2. Random Survival Forest
# --------------------------------------------------------------------------- #
@dataclass
class RandomSurvivalForestModel:
    params: dict = field(default_factory=lambda: {
        "n_estimators": 300,
        "min_samples_leaf": 8,          # regularize hard on 316 rows
        "max_features": "sqrt",
        "n_jobs": -1,
        "random_state": config.RANDOM_STATE,
    })
    _model: object = field(default=None, repr=False)

    def fit(self, X, event, time):
        from sksurv.ensemble import RandomSurvivalForest

        self._model = RandomSurvivalForest(**self.params)
        self._model.fit(np.asarray(X, dtype=float), to_structured_y(event, time))
        return self

    def predict_hit_prob(self, X, horizons=config.HORIZONS) -> np.ndarray:
        fns = self._model.predict_survival_function(np.asarray(X, dtype=float))
        return _survfuncs_to_hit_prob(fns, horizons)


# --------------------------------------------------------------------------- #
# 3. Gradient Boosted Survival Trees
# --------------------------------------------------------------------------- #
@dataclass
class GradientBoostedSurvivalModel:
    params: dict = field(default_factory=lambda: {
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 2,
        "subsample": 0.8,
        "random_state": config.RANDOM_STATE,
    })
    _model: object = field(default=None, repr=False)

    def fit(self, X, event, time):
        from sksurv.ensemble import GradientBoostingSurvivalAnalysis

        self._model = GradientBoostingSurvivalAnalysis(**self.params)
        self._model.fit(np.asarray(X, dtype=float), to_structured_y(event, time))
        return self

    def predict_hit_prob(self, X, horizons=config.HORIZONS) -> np.ndarray:
        fns = self._model.predict_survival_function(np.asarray(X, dtype=float))
        return _survfuncs_to_hit_prob(fns, horizons)


MODEL_REGISTRY = {
    "xgb_cox": XGBoostCoxModel,
    "rsf": RandomSurvivalForestModel,
    "gbst": GradientBoostedSurvivalModel,
}


# --------------------------------------------------------------------------- #
# Optuna hyperparameter tuning (optional)
# --------------------------------------------------------------------------- #
def tune_xgb_cox(X, event, time, n_trials: int = 50):
    """Tune XGBoost Cox with Optuna, maximizing cross-validated concordance.

    Returns the best params dict. Requires optuna + scikit-survival.
    """
    import optuna
    from sklearn.model_selection import KFold
    from sksurv.metrics import concordance_index_censored

    X = np.asarray(X, dtype=float)
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            "objective": "survival:cox",
            "eval_metric": "cox-nloglik",
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 4),
            "min_child_weight": trial.suggest_int("min_child_weight", 3, 12),
            "subsample": trial.suggest_float("subsample", 0.6, 0.95),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.95),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 5.0, log=True),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 2.0, log=True),
            "tree_method": "hist",
            "seed": config.RANDOM_STATE,
        }
        n_estimators = trial.suggest_int("n_estimators", 200, 600, step=50)

        cv = KFold(n_splits=config.N_SPLITS, shuffle=True, random_state=config.RANDOM_STATE)
        scores = []
        for tr, va in cv.split(X):
            m = XGBoostCoxModel(params=params, n_estimators=n_estimators)
            m.fit(X[tr], event[tr], time[tr])
            # Rank by 72h hit prob; concordance wants a risk score.
            risk = m.predict_hit_prob(X[va], horizons=(72,))[:, 0]
            c = concordance_index_censored(event[va].astype(bool), time[va], risk)[0]
            scores.append(c)
        return float(np.mean(scores))

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params

### Cross-validated ensemble
Per-horizon blend weights learned on out-of-fold predictions.

In [ ]:
class SurvivalEnsemble:
    def __init__(self, model_names=("xgb_cox", "rsf", "gbst"), model_params=None):
        self.model_names = list(model_names)
        self.model_params = model_params or {}
        self.weights_ = None            # (n_models, n_horizons)
        self.fitted_models_ = {}

    # ------------------------------------------------------------------ #
    def _new_model(self, name):
        cls = MODEL_REGISTRY[name]
        return cls(**self.model_params.get(name, {}))

    # ------------------------------------------------------------------ #
    def fit(self, X, event, time, horizons=config.HORIZONS):
        X = np.asarray(X, dtype=float)
        oof = self._out_of_fold_predictions(X, event, time, horizons)
        self.weights_ = self._optimize_weights(oof, event, time, horizons)

        # Refit every base model on all the data for final inference.
        for name in self.model_names:
            self.fitted_models_[name] = self._new_model(name).fit(X, event, time)
        return self

    # ------------------------------------------------------------------ #
    def _out_of_fold_predictions(self, X, event, time, horizons):
        """Return {model_name: OOF array (n_samples, n_horizons)}."""
        cv = KFold(n_splits=config.N_SPLITS, shuffle=True, random_state=config.RANDOM_STATE)
        oof = {n: np.zeros((len(X), len(horizons))) for n in self.model_names}
        for tr, va in cv.split(X):
            for name in self.model_names:
                m = self._new_model(name).fit(X[tr], event[tr], time[tr])
                oof[name][va] = m.predict_hit_prob(X[va], horizons)
        return oof

    # ------------------------------------------------------------------ #
    def _optimize_weights(self, oof, event, time, horizons):
        """Grid-search the simplex per horizon; maximize OOF concordance."""
        n_models = len(self.model_names)
        weights = np.zeros((n_models, len(horizons)))
        grid = self._simplex_grid(n_models, step=0.1)

        for j, _h in enumerate(horizons):
            preds = np.stack([oof[n][:, j] for n in self.model_names], axis=1)  # (n, n_models)
            best_c, best_w = -np.inf, np.ones(n_models) / n_models
            for w in grid:
                blend = preds @ w
                c = concordance_index_censored(event.astype(bool), time, blend)[0]
                if c > best_c:
                    best_c, best_w = c, w
            weights[:, j] = best_w
        return weights

    @staticmethod
    def _simplex_grid(n_models, step=0.1):
        """All weight vectors on the simplex with the given step (sum to 1)."""
        ticks = int(round(1.0 / step))
        combos = []
        for raw in product(range(ticks + 1), repeat=n_models):
            if sum(raw) == ticks:
                combos.append(np.array(raw, dtype=float) / ticks)
        return combos

    # ------------------------------------------------------------------ #
    def predict_hit_prob(self, X, horizons=config.HORIZONS) -> np.ndarray:
        if self.weights_ is None:
            raise RuntimeError("Call fit() before predict_hit_prob().")
        X = np.asarray(X, dtype=float)
        per_model = {n: m.predict_hit_prob(X, horizons) for n, m in self.fitted_models_.items()}
        out = np.zeros((len(X), len(horizons)))
        for j in range(len(horizons)):
            blend = np.stack([per_model[n][:, j] for n in self.model_names], axis=1)
            out[:, j] = blend @ self.weights_[:, j]
        return np.clip(out, 0.0, 1.0)

    # ------------------------------------------------------------------ #
    def weight_table(self, horizons=config.HORIZONS):
        """Human-readable weights for docs/README (model x horizon)."""
        import pandas as pd
        return pd.DataFrame(
            self.weights_,
            index=self.model_names,
            columns=[f"{h}h" for h in horizons],
        )

### Per-horizon calibration
The biggest single win: isotonic vs. Platt per horizon, chosen by Brier score.

In [ ]:
def binary_target_by_horizon(event, time, horizon) -> np.ndarray:
    """1 if the fire actually hit a zone at or before ``horizon``, else 0.

    Note the censoring subtlety: a censored fire (event==0) is treated as "not
    hit within 72h", which is correct because the horizons never exceed 72h.
    """
    event = np.asarray(event).astype(bool)
    time = np.asarray(time, dtype=float)
    return ((event) & (time <= horizon)).astype(int)


class _PlattCalibrator:
    """Logistic (sigmoid) calibration on a single probability column."""
    def __init__(self):
        self.lr = LogisticRegression(max_iter=1000)

    def fit(self, p, y):
        self.lr.fit(np.asarray(p).reshape(-1, 1), y)
        return self

    def transform(self, p):
        return self.lr.predict_proba(np.asarray(p).reshape(-1, 1))[:, 1]


class _IsotonicCalibrator:
    def __init__(self):
        self.iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)

    def fit(self, p, y):
        self.iso.fit(np.asarray(p), y)
        return self

    def transform(self, p):
        return self.iso.transform(np.asarray(p))


class PerHorizonCalibrator:
    """Chooses and applies the best calibrator for each horizon."""
    def __init__(self, horizons=config.HORIZONS):
        self.horizons = list(horizons)
        self.calibrators_ = {}          # horizon -> fitted calibrator
        self.method_ = {}               # horizon -> "isotonic" | "platt" | "none"

    def fit(self, oof_probs, event, time):
        """Fit on out-of-fold ensemble probabilities (n_samples, n_horizons)."""
        oof_probs = np.asarray(oof_probs, dtype=float)
        for j, h in enumerate(self.horizons):
            p = oof_probs[:, j]
            y = binary_target_by_horizon(event, time, h)
            self.calibrators_[h], self.method_[h] = self._pick_best(p, y)
        return self

    @staticmethod
    def _pick_best(p, y):
        # Degenerate target (all one class) -> skip calibration for this horizon.
        if len(np.unique(y)) < 2:
            return None, "none"

        p_tr, p_va, y_tr, y_va = train_test_split(
            p, y, test_size=0.3, random_state=config.RANDOM_STATE, stratify=y
        )
        candidates = {
            "isotonic": _IsotonicCalibrator().fit(p_tr, y_tr),
            "platt": _PlattCalibrator().fit(p_tr, y_tr),
        }
        scored = {
            name: brier_score_loss(y_va, cal.transform(p_va))
            for name, cal in candidates.items()
        }
        best = min(scored, key=scored.get)
        # Refit the winner on all the data before returning.
        refit = (_IsotonicCalibrator() if best == "isotonic" else _PlattCalibrator()).fit(p, y)
        return refit, best

    def transform(self, probs) -> np.ndarray:
        probs = np.asarray(probs, dtype=float)
        out = probs.copy()
        for j, h in enumerate(self.horizons):
            cal = self.calibrators_.get(h)
            if cal is not None:
                out[:, j] = cal.transform(probs[:, j])
        return np.clip(out, 0.0, 1.0)

    def summary(self):
        """Which method won at each horizon (for the write-up)."""
        return {f"{h}h": self.method_[h] for h in self.horizons}

### Evaluation utilities
Concordance + Brier + reliability curves for local validation before spending a submission.

In [ ]:
def concordance_by_horizon(probs, event, time, horizons=config.HORIZONS) -> dict:
    """Harrell's C-index at each horizon (uses that horizon's hit-prob as risk)."""
    from sksurv.metrics import concordance_index_censored

    event = np.asarray(event).astype(bool)
    time = np.asarray(time, dtype=float)
    out = {}
    for j, h in enumerate(horizons):
        c = concordance_index_censored(event, time, probs[:, j])[0]
        out[f"c_index_{h}h"] = round(float(c), 5)
    out["c_index_mean"] = round(float(np.mean(list(out.values()))), 5)
    return out


def brier_by_horizon(probs, event, time, horizons=config.HORIZONS) -> dict:
    """Brier score at each horizon (lower is better). Measures calibration+sharpness."""
    out = {}
    vals = []
    for j, h in enumerate(horizons):
        y = binary_target_by_horizon(event, time, h)
        if len(np.unique(y)) < 2:
            out[f"brier_{h}h"] = None
            continue
        b = brier_score_loss(y, probs[:, j])
        out[f"brier_{h}h"] = round(float(b), 5)
        vals.append(b)
    out["brier_mean"] = round(float(np.mean(vals)), 5) if vals else None
    return out


def reliability_curve(probs_h, y_h, n_bins=10):
    """Return (mean_predicted, observed_rate) per bin for a reliability diagram."""
    probs_h = np.asarray(probs_h, dtype=float)
    y_h = np.asarray(y_h, dtype=int)
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(probs_h, bins) - 1
    mean_pred, obs_rate = [], []
    for b in range(n_bins):
        mask = idx == b
        if mask.sum() == 0:
            continue
        mean_pred.append(probs_h[mask].mean())
        obs_rate.append(y_h[mask].mean())
    return np.array(mean_pred), np.array(obs_rate)


def evaluate_all(probs, event, time, horizons=config.HORIZONS) -> dict:
    """Convenience: concordance + Brier in one dict for logging."""
    return {**concordance_by_horizon(probs, event, time, horizons),
            **brier_by_horizon(probs, event, time, horizons)}

### Pipeline helpers
Feature prep, physical reachability guardrail, and monotonicity enforcement.

In [ ]:
# --------------------------------------------------------------------------- #
def load_data(data_dir: Path):
    train = pd.read_csv(data_dir / config.TRAIN_FILE)
    test = pd.read_csv(data_dir / config.TEST_FILE)
    return train, test


def prepare_features(train: pd.DataFrame, test: pd.DataFrame):
    """Engineer features on both splits and align the model-input columns."""
    train_fe = add_engineered_features(train)
    test_fe = add_engineered_features(test)

    feature_cols = [
        c for c in engineered_feature_columns(train_fe)
        if c in test_fe.columns                       # only features present in both
    ]
    X_train = train_fe[feature_cols].fillna(0.0).to_numpy(dtype=float)
    X_test = test_fe[feature_cols].fillna(0.0).to_numpy(dtype=float)
    event = train_fe[config.EVENT_COL].to_numpy()
    time = train_fe[config.TIME_COL].to_numpy(dtype=float)
    return X_train, X_test, event, time, feature_cols


def enforce_monotonicity(probs: np.ndarray) -> np.ndarray:
    """A fire only gets closer over time: prob_12h <= prob_24h <= ... <= prob_72h."""
    return np.maximum.accumulate(probs, axis=1)


def zero_out_impossible_hits(test_df: pd.DataFrame, probs: np.ndarray) -> np.ndarray:
    """Hard prior from the data: no training fire ever hit from beyond ~5 km.

    For fires that are physically too far to reach a zone within a horizon (even
    at a generous spread speed), clamp that horizon's probability toward zero.
    This encodes the distant-fire insight as a guardrail on top of calibration.
    """
    probs = probs.copy()
    d = test_df.get("distance_to_zone_km")
    v = test_df.get("spread_rate_kmh")
    if d is None or v is None:
        return probs
    d = d.to_numpy(dtype=float)
    v = np.maximum(v.to_numpy(dtype=float), 1e-6)
    for j, h in enumerate(config.HORIZONS):
        # Max reachable distance in h hours, with a 2x safety margin on speed.
        unreachable = (2.0 * v * h) < (d - config.HIT_RADIUS_KM)
        probs[unreachable, j] = 0.0
    return enforce_monotonicity(probs)


# --------------------------------------------------------------------------- #


# --------------------------------------------------------------------------- #

## Run it: data → submission
Loads the data, fits the ensemble, calibrates on out-of-fold predictions, applies guardrails, and writes a
valid `submission.csv` to `/kaggle/working`.

In [ ]:
import numpy as np, pandas as pd

train = pd.read_csv(config.DATA_DIR / config.TRAIN_FILE)
test  = pd.read_csv(config.DATA_DIR / config.TEST_FILE)
print("train:", train.shape, "| test:", test.shape)

X_train, X_test, event, time, feature_cols = prepare_features(train, test)
print(f"{len(feature_cols)} model-input features")

ensemble = SurvivalEnsemble().fit(X_train, event, time)
print("per-horizon weights:")
print(ensemble.weight_table().round(2).to_string())

In [ ]:
# Fit calibration on out-of-fold blended predictions (honest, no leakage).
oof = ensemble._out_of_fold_predictions(X_train, event, time, config.HORIZONS)
oof_blend = np.zeros((len(X_train), len(config.HORIZONS)))
for j in range(len(config.HORIZONS)):
    stack = np.stack([oof[n][:, j] for n in ensemble.model_names], axis=1)
    oof_blend[:, j] = stack @ ensemble.weights_[:, j]

calibrator = PerHorizonCalibrator().fit(oof_blend, event, time)
print("calibration per horizon:", calibrator.summary())
print("pre :", evaluate_all(oof_blend, event, time))
print("post:", evaluate_all(calibrator.transform(oof_blend), event, time))

In [ ]:
# Predict on test, calibrate, apply guardrails, enforce monotonicity.
test_probs = ensemble.predict_hit_prob(X_test, config.HORIZONS)
test_probs = calibrator.transform(test_probs)
test_probs = zero_out_impossible_hits(test, test_probs)
test_probs = enforce_monotonicity(test_probs)

submission = pd.DataFrame({config.ID_COL: test[config.ID_COL].to_numpy()})
for j, h in enumerate(config.HORIZONS):
    submission[f"prob_{h}h"] = test_probs[:, j]

submission.to_csv("/kaggle/working/submission.csv", index=False)
print("wrote", len(submission), "rows")
submission.head()

## Recap
Survival framing (uses the censored labels), a CV-weighted 3-model ensemble, **per-horizon calibration**
(the decisive fix for distant-fire overconfidence), and physical guardrails took the baseline from
**0.87397 → 0.96366**, clearing the 0.90 target.